> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验六：使用四类策略优化RMSNorm算子


建议学时：1学时


# 实验任务


## 任务描述


本实验说明RMSNorm优化版如何在基础版之上改造。优化版保持输入输出和计算公式不变，只调整设备侧的数据搬运与计算组织，以减少重复访存和逐元素循环带来的开销。


## 学习目标


能够从基础版的两次全局内存读取和标量累加中识别性能瓶颈；理解优化版采用的分块划分、局部缓存、向量计算和连续搬运四类策略，并能对应到具体代码改动。


# 任务准备


## 优化前的瓶颈与策略总览


优化动机：基础版为了便于验证，先逐元素读取输入累加平方和，再逐元素重新读取输入和权重并写出结果。对于隐藏特征长度为1024的常用形状，这意味着输入被读取两次，平方和采用串行累加，且每次访问都很细碎。优化版的目标不是改变RMSNorm的结果，而是在保持同一输入输出语义的前提下，减少全局内存的细粒度访问，并将规则的逐元素计算交给设备侧向量指令完成。


## 基础版与优化版的前后对比


优化版不改变公式和框架调用入口，而是将基础版的逐元素、两次读取路径替换为“整行搬入—本地向量计算—整行写回”的路径。下表给出每项策略对应的基础版位置、优化版改动和预期作用。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">基础版的实现</th>
<th style="text-align:left;">优化版的修改位置</th>
<th style="text-align:left;">改动带来的作用</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">任务分块</td>
<td style="text-align:left;">在Process中按行循环，每个计算核心处理连续行。</td>
<td style="text-align:left;">保留coreId、rowBegin和rowEnd的行级划分；不改变任务边界。</td>
<td style="text-align:left;">保证接口和任务分工一致，使性能差异来自内核数据流。</td>
</tr>
<tr>
<td style="text-align:left;">局部缓存与连续搬运</td>
<td style="text-align:left;">两次循环中均通过GetValue从全局内存逐元素读取输入。</td>
<td style="text-align:left;">新增CopyIn：用DataCopy将整行输入和共享权重搬入局部张量；新增CopyOut连续写回结果。</td>
<td style="text-align:left;">减少细粒度全局访存，避免第二次逐元素读取输入。</td>
</tr>
<tr>
<td style="text-align:left;">向量计算与归约</td>
<td style="text-align:left;">在两个for循环中完成平方累加、缩放、乘权重和写回。</td>
<td style="text-align:left;">新增Compute：用Mul计算平方，用ReduceSum求和，用Muls缩放，再用Mul乘权重。</td>
<td style="text-align:left;">以整行向量指令替代逐元素循环，缩短归约和逐元素计算路径。</td>
</tr>
<tr>
<td style="text-align:left;">队列化的数据生命周期</td>
<td style="text-align:left;">基础版直接读取和写出，没有局部对象的显式生命周期。</td>
<td style="text-align:left;">新增输入、权重、输出队列和三个临时缓冲，并按CopyIn、Compute、CopyOut组织。当前队列深度为1。</td>
<td style="text-align:left;">明确数据所有权，为后续双缓冲预留结构；当前实现不宣称已隐藏搬运时间。</td>
</tr>
</tbody></table>


基础版每处理一行数据，都要两次从全局内存读取输入，并使用逐元素循环计算平方和。这种方式便于验证，却会产生大量细粒度访存和串行累加。优化版将数据整行搬入片上缓存，在本地完成平方、求和、缩放和权重相乘，再一次性写回结果。


图 1  RMSNorm优化版的数据流


## 算子定义与接口约定


优化版沿用基础版的输入输出约定和数学公式，因此两者可以在相同输入下直接比较。不同之处只在设备侧的实现路径。


公式、权重布局和稳定项均与基础版一致；优化只改变数据如何搬运、如何在本地计算，以及计算任务如何分配给各个计算核心。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">input、weight、output均为float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">连续Tensor；最后一维必须等于weight.numel()</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">rows = input.numel() / hidden，支持一维、二维及更高维输入</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum = min(8, rows)，rowsPerCore向上取整</td>
</tr>
<tr>
<td style="text-align:left;">优化版附加约束</td>
<td style="text-align:left;">hidden必须是8个float的倍数，保证GM↔UB对齐搬运</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">rows=128，hidden=1024，eps=1e-6，blockDim=8</td>
</tr>
</tbody></table>


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置/说明</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0；需先source set_env.sh</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extension，torch.ops.rmsnorm_custom.rms_norm</td>
</tr>
<tr>
<td style="text-align:left;">构建结果</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、librmsnorm_torch_register.so；out/bin/rmsnorm_*_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">PyTorch torch.rsqrt(input.pow(2).mean(...)+eps) 与standalone C++ reference</td>
</tr>
</tbody></table>


# 任务实施


本环节按照“确定优化目标—改写设备端数据流—保持框架接口—构建加载—正确性验证—性能计时与结果分析”六步推进。每一步都与基础版建立对应关系，确保性能变化来自内核优化，而不是接口、输入或计时口径变化。


## 步骤一：确定瓶颈、优化目标与对比基线


本步从基础版重复读取、标量累加和细粒度访存出发，固定输入形状、接口和计时口径，形成可公平比较的优化基线。


优化版的四项策略分别对应任务分块、局部缓存与连续搬运、向量计算与归约、队列化的数据生命周期。它们共同针对基础版的重复读取、串行累加和细粒度全局访存；具体改动可与基础版的Process函数逐项对照阅读。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">策略</th>
<th style="text-align:left;">为什么使用</th>
<th style="text-align:left;">本工程的具体实施</th>
<th style="text-align:left;">代码/约束</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">Tiling</td>
<td style="text-align:left;">rows可能大于可用AI Core数，hidden也必须在本地缓冲容量内被有界处理；需要把运行时shape映射为稳定的核心分工并覆盖尾行。</td>
<td style="text-align:left;">Host填充rows、hidden、coreNum、rowsPerCore、eps、invHidden。coreNum=min(8, rows)，rowsPerCore=(rows+coreNum-1)/coreNum；每个core处理连续的若干完整行，rowEnd截断最后一个core。</td>
<td style="text-align:left;">coreId、rowBegin、rowEnd；标准形状128×1024时为8个core、每core 16行。</td>
</tr>
<tr>
<td style="text-align:left;">Vectorization</td>
<td style="text-align:left;">平方、归约、缩放、权重乘法都是连续FP32向量上的规则操作；使用AscendC向量/归约指令可消除基础版的标量GetValue/SetValue循环。</td>
<td style="text-align:left;">在UB的LocalTensor上使用Mul(x,x)生成平方ReduceSum计算一行平方和，Muls完成统一scale，Mul叠加weight；输出以连续块回写。</td>
<td style="text-align:left;">Mul、ReduceSum、Muls、Mul，长度均为hidden。</td>
</tr>
<tr>
<td style="text-align:left;">Pipeline</td>
<td style="text-align:left;">搬运、计算、写回属于不同阶段。以队列与buffer明确生命周期，可避免同一LocalTensor被错误复用，并为后续双缓冲、搬运/计算重叠保留接口。</td>
<td style="text-align:left;">TPipe分配VECIN的inputQue/weightQue、VECOUT的outputQue、VECCALC的square/reduce/sum buffer。每行按CopyIn→Compute→CopyOut执行，EnQue/DeQue/FreeTensor串联阶段。</td>
<td style="text-align:left;">当前TQue深度均为1：这是队列化流水组织，尚未实现跨tile的双缓冲重叠；性能解释不可把它写成已完全隐藏搬运时间。</td>
</tr>
<tr>
<td style="text-align:left;">Access &amp; Layout</td>
<td style="text-align:left;">基础版逐标量从GM读取且input被二次读取；连续合并搬运到UB后，计算可在本地完成，从而减少细粒度GM访问。</td>
<td style="text-align:left;">DataCopy将一整行input与连续weight从GM搬入LocalTensor；中间square、reduce、output均留在UB/VECCALC/VECOUT，最终DataCopy连续回写output。优化wrapper与standalone拒绝hidden%8!=0，满足对齐GM↔UB copy条件。</td>
<td style="text-align:left;">input/weight/output为FP32连续布局；优化版强制hidden为8个float的倍数。</td>
</tr>
</tbody></table>


## 步骤二：改写设备端数据搬运与向量计算


本步将整行数据搬入片上缓存，使用向量运算完成平方、归约、缩放和加权，并一次性写回结果。


```text
// Init：Pipeline的队列/缓冲区分配（当前深度为 1）
```


```text
const uint32_t bytes = hidden_ * sizeof(float);
```


```text
pipe_.InitBuffer(inputQue_, 1, bytes);
```


```text
pipe_.InitBuffer(weightQue_, 1, bytes);
```


```text
pipe_.InitBuffer(outputQue_, 1, bytes);
```


```text
pipe_.InitBuffer(squareBuf_, bytes);
```


```text
pipe_.InitBuffer(reduceBuf_, bytes);
```


```text
pipe_.InitBuffer(sumBuf_, 32);
```


```text
// Access & Layout：连续GM→UB搬运
```


```text
DataCopy(xLocal, inputGm_[row * hidden_], hidden_);
```


```text
DataCopy(wLocal, weightGm_[0], hidden_);
```


```text
// Vectorization：UB内的平方、归约、缩放与逐元素权重乘法
```


```text
Mul(squareLocal, xLocal, xLocal, static_cast<int32_t>(hidden_));
```


```text
ReduceSum(sumLocal, squareLocal, reduceLocal, static_cast<int32_t>(hidden_));
```


```text
const float scale = RmsNormInvSqrtApprox(sumLocal.GetValue(0) * invHidden_ + eps_);
```


```text
Muls(yLocal, xLocal, scale, static_cast<int32_t>(hidden_));
```


```text
Mul(yLocal, yLocal, wLocal, static_cast<int32_t>(hidden_));
```


```text
DataCopy(outputGm_[row * hidden_], yLocal, hidden_);
```


## 步骤三：保持主机侧调用与算子注册一致


本步保持输入输出、Tiling语义和框架调用入口不变，使基础版与优化版能够在同一上层代码中替换比较。


为保证比较公平，优化版沿用基础版的调用入口、输入输出约定和模型中的替换位置。这样，性能差异可以归因于设备侧实现，而不是调用方式变化。


## 步骤四：构建优化版并加载动态库


本步生成优化版设备库、注册库和独立测试程序，并确认运行时依赖与动态库路径正确。


以下命令用于构建优化版并运行验证。基础版与优化版使用相同的框架调用名称，比较时应在不同的Python进程中执行，避免重复加载。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops/RmsNormOptimizedExperiment
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH
```


```bash
bash scripts/check_env.sh
```


```bash
bash scripts/build.sh
```


```bash
python3 tests/test_torch_op.py
```


## 步骤五：Golden数据与模型接入正确性验证


本步以同一参考公式验证一维、二维和三维输入，并通过模型级测试确认优化没有改变权重、稳定项和调用位置。


优化版沿用基础版的正确性测试：对一维、二维和三维输入分别与参考结果比较，确认优化前后在允许误差范围内保持一致。


```python
def reference(input_tensor, weight, eps):
```


```text
variance = input_tensor.pow(2).mean(dim=-1, keepdim=True)
```


```python
return input_tensor * torch.rsqrt(variance + eps) * weight
```


```text
golden = reference(input_tensor, weight, eps)
```


```python
actual = torch.ops.rmsnorm_custom.rms_norm(input_tensor, weight, eps)
```


```text
diff = (actual - golden).abs()
```


```python
ok = torch.allclose(actual, golden, atol=atol, rtol=rtol)
```


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">版本</th>
<th style="text-align:left;">单测覆盖结果</th>
<th style="text-align:left;">判定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">基础版</td>
<td style="text-align:left;">(1024,)、(128,1024)、(2,4,1024)均PASS；最大绝对误差不超过7.15255737e-07</td>
<td style="text-align:left;">PASS</td>
</tr>
<tr>
<td style="text-align:left;">优化版</td>
<td style="text-align:left;">同三种shape均PASS；最大绝对误差不超过2.86102295e-06</td>
<td style="text-align:left;">PASS</td>
</tr>
</tbody></table>


工程还提供模型级对比测试，用于确认优化后仍保持原有的权重、稳定项和调用位置。该测试用于验证接入正确性，不替代单算子的设备侧计时。


## 步骤六：性能计时、结果记录与收益分析


本步在与基础版相同的形状、核心数、预热和重复设置下统计设备侧时间，并结合实测结果解释优化收益与边界。


独立测试程序使用固定输入，先预热，再重复启动内核，并通过设备事件计算单次执行时间。计时不包含随机数据生成、主机与设备之间的初始数据传输、结果比对和Python调用。


```bash
cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops/Qwen2.5cann_ops/RmsNormOptimizedExperiment
```


```bash
source /home/developer/Ascend/cann-8.5.2/set_env.sh
```


```bash
export LD_LIBRARY_PATH=$PWD/out/lib:$LD_LIBRARY_PATH
```


```bash
out/bin/rmsnorm_optimized_standalone \
```


```text
--rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5 --eps 1e-6
```


```text
// standalone中的统计核心
```


```text
aclrtRecordEvent(start, stream);
```


```text
for (uint32_t i = 0; i < opt.repeat; ++i) {
```


```text
ACLRT_LAUNCH_KERNEL(rmsnorm_optimized_kernel)(blockDim, stream, ...);
```


```text
}
```


```text
aclrtRecordEvent(stop, stream);
```


```text
aclrtEventElapsedTime(&elapsedMs, start, stop);
```


```text
const double us = elapsedMs * 1000.0 / opt.repeat;
```


## 实测结果与解释


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">版本</th>
<th style="text-align:left;">mean (us)</th>
<th style="text-align:left;">median (us)</th>
<th style="text-align:left;">min / max (us)</th>
<th style="text-align:left;">max_abs / mean_abs</th>
<th style="text-align:left;">正确性</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">基础版</td>
<td style="text-align:left;">491.313</td>
<td style="text-align:left;">491.275</td>
<td style="text-align:left;">491.262 / 491.403</td>
<td style="text-align:left;">7.15255737e-07 / 2.35556367e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
<tr>
<td style="text-align:left;">优化版</td>
<td style="text-align:left;">9.262</td>
<td style="text-align:left;">9.266</td>
<td style="text-align:left;">9.232 / 9.284</td>
<td style="text-align:left;">2.86102295e-06 / 1.29399225e-07</td>
<td style="text-align:left;">PASS</td>
</tr>
</tbody></table>


本节数据来自项目测试日志。基础版和优化版使用同一输入形状、相同的计算核心数量及相同的预热和重复设置；因此性能差异可以反映优化版对内核数据流的改进效果，但不能直接等同于完整模型的加速倍数。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">基础版的问题</th>
<th style="text-align:left;">优化版如何改动</th>
<th style="text-align:left;">对应代码</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">任务分块</td>
<td style="text-align:left;">行数通常多于可用计算核心，需要合理分工并处理最后不足一块的数据。</td>
<td style="text-align:left;">主机侧计算核心数和每个核心负责的行数；最后一个核心按实际行数结束。</td>
<td style="text-align:left;">coreId、rowBegin、rowEnd</td>
</tr>
<tr>
<td style="text-align:left;">向量计算</td>
<td style="text-align:left;">基础版逐元素读取、累加和写出，循环次数多。</td>
<td style="text-align:left;">在片上缓存中一次处理整行数据：平方、求和、缩放和权重相乘均使用向量指令。</td>
<td style="text-align:left;">Mul、ReduceSum、Muls</td>
</tr>
<tr>
<td style="text-align:left;">局部缓存</td>
<td style="text-align:left;">输入在两次扫描中反复从全局内存读取。</td>
<td style="text-align:left;">输入、权重和输出分别进入片上缓存，中间结果保留在本地，减少细粒度全局访问。</td>
<td style="text-align:left;">输入、权重、输出队列</td>
</tr>
<tr>
<td style="text-align:left;">连续搬运</td>
<td style="text-align:left;">逐元素访问难以充分利用连续数据通路。</td>
<td style="text-align:left;">将一整行输入和权重连续搬入，计算完成后连续写回；要求隐藏特征长度满足对齐条件。</td>
<td style="text-align:left;">DataCopy与对齐检查</td>
</tr>
</tbody></table>


## 优化的边界与后续工作


优化版不改变计算公式、权重布局和框架调用方式。当前模型级调用仍会产生主机与设备之间的数据传输，因此后续还可以继续减少传输、复用缓冲区，并针对不同输入规模调整分块大小。


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RmsNormOptimizedExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RmsNormOptimizedExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/rmsnorm_optimized_standalone --rows 128 --hidden 1024 --block-dim 8 --warmup 10 --repeat 50 --rounds 5 --eps 1e-6


# 常见问题与排查


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">现象</th>
<th style="text-align:left;">优先检查</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">torch.ops.rmsnorm_custom.rms_norm不存在</td>
<td style="text-align:left;">确认已bash scripts/build.sh，LD_LIBRARY_PATH包含out/lib，且register .so被load_torch_ops()加载</td>
</tr>
<tr>
<td style="text-align:left;">数值误差超过阈值</td>
<td style="text-align:left;">核对input最后一维、weight长度、FP32、eps、连续布局和tiling中invHidden</td>
</tr>
<tr>
<td style="text-align:left;">优化版直接报错</td>
<td style="text-align:left;">hidden是否为8的倍数；该限制来自对齐的GM↔UB DataCopy</td>
</tr>
<tr>
<td style="text-align:left;">性能异常</td>
<td style="text-align:left;">确认warmup后才计时；使用相同blockDim、shape、repeat、rounds；不要把wrapper/拷贝时间与standalone device时间混合</td>
</tr>
<tr>
<td style="text-align:left;">同进程比较两版失败</td>
<td style="text-align:left;">两版注册同一rmsnorm_custom namespace，应使用独立Python进程</td>
</tr>
</tbody></table>


# 实验总结


本实验完成了RMSNorm优化版从公式、分块参数、AscendC内核、ACL/PyTorch注册到单元测试和独立性能验证的完整链路。优化策略均已对应到实际代码，并在同一输入形状和计时口径下验证其收益。
